# MLP simples com a diferença


- 1 saida - Theta
- 1 entradas - Vd - Ve
- 1 camada
- 1 neurônio


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

Datasets = []
PREDICTORS = "VD-VE"   
TARGET = "Theta"       

for i in range(4):   
    Dataset = pd.read_csv(f"../Dados/Data{i + 1}.csv")
    
    # Remove colunas desnecessárias
    Dataset = Dataset.drop(columns=["X", "Y", "Setpoint VD", "Setpoint VE", "tempo"])
    
    # Ajusta índice pelo tempo
    Dataset.index = (np.arange(0, len(Dataset), 1).astype(float) * 0.07).round(5)
    
    # Cria nova coluna com VD - VE
    Dataset[PREDICTORS] = Dataset["VD"] - Dataset["VE"]
    
    Datasets.append(Dataset)
    
    print(f"++++++++++++++++++++ Dataset {i+1} +++++++++++++++++++++++")
    print(Dataset.head(5))


In [ ]:
for  i in range(4):
    Dataset = Datasets[i].copy()  # evita sobrescrever o original
    Dataset["DeltaTheta"] = Dataset[TARGET].shift(-1) - Dataset[TARGET]
    # ou equivalente: Dataset["DeltaTheta"] = Dataset[TARGET].diff().shift(-1)
    
    # remove o último NaN (último ponto não tem próximo valor)
    Dataset = Dataset.dropna(subset=["DeltaTheta"])
    Datasets[i] = Dataset
    print(f"++++++++++++++++++++ Dataset {i+1} +++++++++++++++++++++++")
    print(Datasets[i].tail(5))


In [ ]:
NormDatasets = []

PREDICTORS = ["VD-VE" ]  
TARGET = ["DeltaTheta"]
SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

TrainDataset = Datasets[0]
TrainDataset[PREDICTORS] = SCALER.fit_transform(TrainDataset[PREDICTORS])
TrainDataset[TARGET] = OUT_SCALER.fit_transform(TrainDataset[TARGET])
NormDatasets.append(TrainDataset)

for i in range(3):
      CurrentTestDataset = Datasets[i + 1]
      CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
      CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
      NormDatasets.append(CurrentTestDataset)
      print(f"++++++++++++++++++++ Dataset Normalizado {i+1} +++++++++++++++++++++++")
      print(NormDatasets[i].head(5))

In [ ]:
x_train = np.array(TrainDataset[PREDICTORS])
y_train = np.array(TrainDataset[TARGET])

x_val = np.array((NormDatasets[1])[PREDICTORS])
y_val = np.array((NormDatasets[1])[TARGET])

print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

In [ ]:
import matplotlib.pyplot as plt

def PlotHistory(history):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Loss')
    plt.title('Training History')
    plt.legend()
    plt.grid(True)
    plt.show()
    
def PlotOut(axs, title, y_true, y_pred):
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)
    time = (np.arange(0, len(y_pred), 1).astype(float) * 0.07).round(5)

    axs.scatter(time, y_true, marker='o', label='Amostras Reais')
    axs.scatter(time, y_pred, marker='x', label='Valores Preditos')
    axs.set_title(f'{title}')
    axs.set_xlabel('Theta')
    axs.set_ylabel('tempo')
    axs.legend()
    axs.grid(True)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

TITLES = ["Training", "Validation", "Test 1", "Test 2"]

def EvalModel(model, save_if_r2_gt=0.5, fig_name_prefix="model"):
    fig, axs = plt.subplots(4, 1, figsize=(12, 12))
    metrics = {}
    max_r2 = -np.inf  # guarda o maior R² encontrado
    
    for i, dataset in enumerate(NormDatasets):
        x = np.array(dataset[PREDICTORS])
        
        y_diff = OUT_SCALER.inverse_transform(dataset[TARGET])
        y_pred_diff = OUT_SCALER.inverse_transform(model.predict(x))
        
        theta0 = Datasets[i]["Theta"].iloc[0]  
        
        y_real = theta0 + np.cumsum(y_diff, axis=0)
        y_pred = theta0 + np.cumsum(y_pred_diff, axis=0)
        
        # métricas
        R2 = r2_score(y_real, y_pred)  
        MSE = mean_squared_error(y_real, y_pred)
        metrics[f"R²_{TITLES[i]}"] = R2
        metrics[f"MSE_{TITLES[i]}"] = MSE
        
        max_r2 = max(max_r2, R2)  # atualiza máximo
        
        PlotOut(axs[i], TITLES[i], y_real, y_pred)
    
    # Salvar figura se R² > limite
    if max_r2 > save_if_r2_gt:
        fname = f"{fig_name_prefix}_R2_{max_r2:.2f}.png"
        plt.tight_layout()
        plt.savefig(fname, dpi=300)
        print(f"Figura salva em {fname} (R² max {max_r2:.2f})")
    
    plt.close(fig)  # evita abrir um monte de janelas
    
    return metrics


In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from keras.callbacks import EarlyStopping
from keras import initializers

INPUT_SIZE = len(PREDICTORS)

N_MODELS = 300
seeds = np.random.choice(range(1, 1000000), size=N_MODELS, replace=False)

excel_file = "resultados.xlsx"

for i, s in enumerate(seeds):
    initializer = initializers.RandomNormal(seed=int(s))

    # cria modelo
    model = keras.models.Sequential([
        keras.layers.Input(shape=(INPUT_SIZE,)),
        keras.layers.Dense(1, activation="linear", kernel_initializer=initializer),  
    ])

    w0 = model.get_weights()

    model.compile(loss="mean_squared_error", optimizer="adam")
    early_stopping_monitor = EarlyStopping(
        monitor='val_loss',
        patience=50,
        restore_best_weights=True
    )

    history = model.fit(
        x_train, 
        y_train, 
        epochs=1000,
        callbacks=[early_stopping_monitor],
        validation_data=(x_val, y_val),
        verbose=0
    )
    PlotHistory(history)

    wf = model.get_weights()

    metrics = EvalModel(model, save_if_r2_gt=0.5, fig_name_prefix=f"model_{i}")
    
    row = {
        "Model": f"model_{i}",
        **metrics,
        "Seed": s,
        "W0": str([w.round(4).tolist() for w in w0]),
        "Wf": str([w.round(4).tolist() for w in wf])
    }
    display(row)

    df = pd.DataFrame([row])

    # salva/atualiza Excel incrementalmente
    try:
        # tenta abrir existente e adicionar linha
        old = pd.read_excel(excel_file)
        new_df = pd.concat([old, df], ignore_index=True)
        new_df.to_excel(excel_file, index=False)
    except FileNotFoundError:
        # se não existir, cria arquivo novo
        df.to_excel(excel_file, index=False)

    print(f"Modelo {i} treinado e salvo no Excel")
